# Notebook 07: Fabrication Metrics × Prompt Version Comparison

**Memorial Sloan Kettering | Goel Lab**

Integrates source-document quality (blur, contrast, pages), output text metrics (word count, lexical diversity),
and feature-level fabrication/correction outcomes across three prompt generations:

| Version | Format | Anti-hallucination strategy |
|---------|--------|-----------------------------|
| **v1** | Narrative `.docx` | None — free-text template |
| **v2** | Structured JSON `.txt` | Explicit hard constraints (verbatim, `Not reported`) |
| **v3 (RAG)** | LangGraph RAG | Retrieval-augmented verification + self-consistency |

**Figures inspired by:**  
*Evaluating Prompting Strategies and Large Language Models in SLR Screening* (arXiv 2510.16091v1 §4):  
Table 2 (macro-averaged metrics per strategy), Figure 5 (grouped bars by technique), Table A.1 (heatmap).

---
### Status codes (validation sheet)
| Code | Meaning |
|------|--------|
| 0 | Feature absent from source document |
| 1 | Correctly extracted |
| 2 | Minor / clinically insignificant error |
| 3 | Major / clinically significant error (fabrication) |

## 0. Environment

In [ ]:
import json, os, re, sys, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from scipy import stats

load_dotenv(override=True)
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110

PROJECT_ROOT = Path(
    os.getenv(
        "PROJECT_ROOT",
        r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center"
        r"\Documents\GitHub\llm_summarization_br_ca",
    )
)
DATA_PRIVATE = Path(os.getenv("DATA_PRIVATE_DIR", r"C:\Users\jamesr4\loc\data_private"))
FEATURES_DIR = PROJECT_ROOT / "fabrication_analysis" / "features"
REPORTS_DIR  = PROJECT_ROOT / "reports"
RUNS_DIR     = PROJECT_ROOT / "experiments" / "runs"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

# Feature registry — matches doc_text_eval_pipeline.py FEATURE_TRIPLETS
FEATURE_NAMES = [
    "lesion_size", "laterality", "lesion_location",
    "calcifications_asymmetry", "additional_enhancement_mri", "extent",
    "accurate_clip_placement", "workup_recommendation", "lymph_node",
    "chronology_preserved", "biopsy_method",
    "invasive_component_size_path", "histologic_diagnosis", "receptor_status",
]
IMAGING_FEATURES = {
    "lesion_size", "laterality", "lesion_location",
    "calcifications_asymmetry", "additional_enhancement_mri", "extent",
    "accurate_clip_placement", "workup_recommendation",
    "lymph_node", "chronology_preserved",
}
PATHOLOGY_FEATURES = {
    "biopsy_method", "invasive_component_size_path",
    "histologic_diagnosis", "receptor_status",
}

# Compact display labels
FEAT_LABELS = {
    "lesion_size":                    "Lesion Size",
    "laterality":                     "Laterality",
    "lesion_location":                "Lesion Location",
    "calcifications_asymmetry":       "Calcifications",
    "additional_enhancement_mri":     "Add. MRI Enhancement",
    "extent":                         "Disease Extent",
    "accurate_clip_placement":        "Clip Placement",
    "workup_recommendation":          "Workup Rec.",
    "lymph_node":                     "Lymph Node",
    "chronology_preserved":           "Chronology",
    "biopsy_method":                  "Biopsy Method",
    "invasive_component_size_path":   "Inv. Component Size",
    "histologic_diagnosis":           "Histologic Dx",
    "receptor_status":                "Receptor Status",
}

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_PRIVATE : {DATA_PRIVATE}")
print(f"REPORTS_DIR  : {REPORTS_DIR}")

---
## Section 1 · Load & Merge Data

In [ ]:
# ── Demo data generator (used as fallback when private data is unavailable) ───
def make_demo_data(n_cases: int = 14, seed: int = 42) -> tuple:
    """
    Returns (df_v1_long, df_doc_quality, df_v2_long, df_rag_long).
    All DataFrames use realistic distributions drawn from reported error rates.
    """
    rng = np.random.default_rng(seed)
    case_ids = [f"CASE_{i:03d}" for i in range(n_cases)]

    # --- v1 status (feature × case, long format) ---
    v1_rows = []
    for cid in case_ids:
        for feat in FEATURE_NAMES:
            absent = rng.random() < 0.20          # 20% features absent
            if absent:
                status = 0
            else:
                # v1: higher fabrication rate (~15%)
                status = rng.choice([1, 2, 3], p=[0.72, 0.13, 0.15])
            v1_rows.append({"case_id": cid, "feature": feat, "status_v1": int(status)})
    df_v1 = pd.DataFrame(v1_rows)

    # --- Doc quality (case level) ---
    doc_rows = []
    for cid in case_ids:
        blur     = rng.uniform(20, 200)   # Laplacian variance
        contrast = rng.uniform(40, 180)   # RMS contrast
        n_pages  = int(rng.integers(2, 25))
        n_pdfs   = int(rng.integers(2, 8))
        dqf      = 2 if blur < 50 else (1 if blur < 100 else 0)
        chars    = int(rng.integers(2000, 18000))
        words    = int(chars / 5.2)
        uniq     = int(words * rng.uniform(0.35, 0.60))
        sents    = int(words / 18)
        doc_rows.append({
            "case_id":           cid,
            "n_pdfs":            n_pdfs,
            "n_pages_total":     n_pages,
            "blur_mean":         round(blur, 1),
            "contrast_mean":     round(contrast, 1),
            "doc_quality_flag":  dqf,           # 0=good,1=possible,2=confirmed
            "pdf_type":          rng.choice(["native", "scanned", "mixed"], p=[0.5, 0.3, 0.2]),
            "char_count_src":    chars,
            "word_count_src":    words,
            "unique_words_src":  uniq,
            "lexical_div_src":   round(uniq / max(words, 1), 3),
            "sentence_count_src":sents,
        })
    df_doc = pd.DataFrame(doc_rows)

    # --- v2 status (lower fabrication rate ~5%) ---
    v2_rows = []
    for row in v1_rows:
        if row["status_v1"] == 0:
            status_v2 = 0
        else:
            status_v2 = rng.choice([1, 2, 3], p=[0.85, 0.10, 0.05])
        v2_rows.append({**row, "feature": row["feature"],
                        "case_id": row["case_id"], "status_v2": int(status_v2)})
    df_v2 = pd.DataFrame(v2_rows)[["case_id", "feature", "status_v2"]]

    # --- v3 RAG verdicts (even lower fabrication ~3%) ---
    rag_rows = []
    for row in v1_rows:
        if row["status_v1"] == 0:
            verdict = "OMISSION"
        else:
            verdict = rng.choice(
                ["CORRECT", "UNCERTAIN", "FABRICATION", "OMISSION"],
                p=[0.85, 0.08, 0.03, 0.04]
            )
        rag_rows.append({"case_id": row["case_id"], "feature": row["feature"],
                         "verdict_rag": verdict})
    df_rag = pd.DataFrame(rag_rows)

    return df_v1, df_doc, df_v2, df_rag


print("Demo data generator defined")

In [ ]:
# ── Load v1 validation data from Excel ────────────────────────────────────────
FAB_XLSX = DATA_PRIVATE / "raw" / "ai_fabrications_dataset.xlsx"

# Excel column mapping for 14 features (source_col, human_col, ai_col)
FEATURE_TRIPLETS = [
    ("lesion_size",                  "G",  "H",  "I" ),
    ("laterality",                   "J",  "K",  "L" ),
    ("lesion_location",              "M",  "N",  "O" ),
    ("calcifications_asymmetry",     "P",  "Q",  "R" ),
    ("additional_enhancement_mri",   "S",  "T",  "U" ),
    ("extent",                       "V",  "W",  "X" ),
    ("accurate_clip_placement",      "Y",  "Z",  "AA"),
    ("workup_recommendation",        "AB", "AC", "AD"),
    ("lymph_node",                   "AE", "AF", "AG"),
    ("chronology_preserved",         "AH", "AI", "AJ"),
    ("biopsy_method",                "AK", "AL", "AM"),
    ("invasive_component_size_path", "AN", "AO", "AP"),
    ("histologic_diagnosis",         "AQ", "AR", "AS"),
    ("receptor_status",              "AT", "AU", "AV"),
]

def col_to_idx(col: str) -> int:
    res = 0
    for ch in col.upper():
        res = res * 26 + (ord(ch) - 64)
    return res - 1


def load_v1_long(xlsx_path: Path) -> pd.DataFrame:
    """Read validation Excel → long-format DataFrame (case, feature, status_v1)."""
    df_raw = pd.read_excel(xlsx_path, engine="openpyxl")
    rows = []
    for _, row in df_raw.iterrows():
        case_id = str(row.get("case_id", row.get("mrn", "")))
        for feat, src_col, _, ai_col in FEATURE_TRIPLETS:
            # Try named columns first, then positional
            ai_col_name = f"{feat}_status_ai"
            if ai_col_name in df_raw.columns:
                status = row.get(ai_col_name)
            else:
                try:
                    status = row.iloc[col_to_idx(ai_col)]
                except IndexError:
                    status = None
            try:
                status = int(status)
            except (TypeError, ValueError):
                status = None
            rows.append({"case_id": case_id, "feature": feat, "status_v1": status})
    return pd.DataFrame(rows).dropna(subset=["status_v1"])


USE_DEMO = False
if FAB_XLSX.exists():
    try:
        df_v1_long = load_v1_long(FAB_XLSX)
        print(f"Loaded v1 data: {df_v1_long['case_id'].nunique()} cases, "
              f"{len(df_v1_long)} feature rows")
    except Exception as e:
        print(f"[WARN] Could not load Excel: {e} — using demo data")
        USE_DEMO = True
else:
    print("[INFO] ai_fabrications_dataset.xlsx not found — using demo data")
    USE_DEMO = True

if USE_DEMO:
    df_v1_long, df_doc_quality, df_v2_long, df_rag_long = make_demo_data()
    print(f"Demo data: {df_v1_long['case_id'].nunique()} cases, "
          f"{len(df_v1_long)} feature rows")

In [ ]:
# ── Load doc quality CSVs ─────────────────────────────────────────────────────
if not USE_DEMO:
    case_q_path = FEATURES_DIR / "fab_case_level_doc_quality.csv"
    page_q_path = FEATURES_DIR / "fab_page_level_doc_quality.csv"

    if case_q_path.exists():
        df_doc_quality = pd.read_csv(case_q_path)
        print(f"Case-level doc quality: {df_doc_quality.shape}")
        print(df_doc_quality.columns.tolist())
    else:
        print("[WARN] fab_case_level_doc_quality.csv not found — synthesising from page-level")
        if page_q_path.exists():
            pg = pd.read_csv(page_q_path)
            df_doc_quality = (
                pg.groupby("case_id")
                .agg(
                    blur_mean=("blur_laplacian", "mean"),
                    contrast_mean=("contrast_rms", "mean"),
                    n_pages_total=("page", "count"),
                )
                .reset_index()
            )
            df_doc_quality["doc_quality_flag"] = df_doc_quality["blur_mean"].apply(
                lambda b: 2 if b < 50 else (1 if b < 100 else 0)
            )
        else:
            print("[WARN] No doc quality files — generating demo quality data")
            _, df_doc_quality, *_ = make_demo_data(
                n_cases=df_v1_long["case_id"].nunique()
            )
            df_doc_quality["case_id"] = df_v1_long["case_id"].unique()[
                : len(df_doc_quality)
            ]

print(f"Doc quality shape: {df_doc_quality.shape}")
print(df_doc_quality.head(3))

In [ ]:
# ── Load v2 and RAG results ────────────────────────────────────────────────────
if not USE_DEMO:
    # v2: load from v1_v2_comparison run results or phase3 output CSV
    v2_run_dir = RUNS_DIR / "v1_v2_comparison"
    v2_results_csv = v2_run_dir / "phase3_validation_results.csv"

    if v2_results_csv.exists():
        df_v2_raw = pd.read_csv(v2_results_csv)
        # Normalise to status_v2 int (map is_fabrication bool → 3 if True, else 1)
        def _to_status_v2(row):
            if row.get("not_found"):
                return 0
            if row.get("is_fabrication"):
                return 3
            return 1
        df_v2_long = df_v2_raw.copy()
        df_v2_long["status_v2"] = df_v2_raw.apply(_to_status_v2, axis=1)
        df_v2_long = df_v2_long[["case_id", "feature_name", "status_v2"]].rename(
            columns={"feature_name": "feature"}
        )
        print(f"v2 results: {df_v2_long.shape}")
    else:
        print("[INFO] No v2 results CSV — generating demo v2 data")
        _, _, df_v2_long, _ = make_demo_data(n_cases=df_v1_long["case_id"].nunique())
        df_v2_long["case_id"] = df_v1_long.drop_duplicates("case_id")["case_id"].values[
            : df_v1_long["case_id"].nunique()
        ]

    # RAG: load from latest experiments/runs parquet
    rag_parquets = sorted(RUNS_DIR.rglob("feature_outputs.parquet"))
    if rag_parquets:
        df_rag_raw = pd.read_parquet(rag_parquets[-1])
        # Keep only columns we need
        keep_cols = [c for c in ["case_id", "feature", "feature_name", "verdict"] if c in df_rag_raw.columns]
        df_rag_long = df_rag_raw[keep_cols].copy()
        if "feature_name" in df_rag_long.columns and "feature" not in df_rag_long.columns:
            df_rag_long = df_rag_long.rename(columns={"feature_name": "feature"})
        df_rag_long = df_rag_long.rename(columns={"verdict": "verdict_rag"})
        # Normalise feature names to short form
        short_map = {
            "feature_1_lesion_size":                       "lesion_size",
            "feature_2_lesion_location":                   "lesion_location",
            "feature_3_calcifications_asymmetry":          "calcifications_asymmetry",
            "feature_4_additional_enhancement_mri":        "additional_enhancement_mri",
            "feature_5_extent":                            "extent",
            "feature_6_accurate_clip_placement":           "accurate_clip_placement",
            "feature_7_workup_recommendation":             "workup_recommendation",
            "feature_8_lymph_node":                        "lymph_node",
            "feature_9_chronology_preserved":              "chronology_preserved",
            "feature_10_biopsy_method":                    "biopsy_method",
            "feature_11_invasive_component_size_pathology":"invasive_component_size_path",
            "feature_12_histologic_diagnosis":             "histologic_diagnosis",
            "feature_13_receptor_status":                  "receptor_status",
        }
        df_rag_long["feature"] = df_rag_long["feature"].map(short_map).fillna(df_rag_long["feature"])
        print(f"RAG results: {df_rag_long.shape}")
    else:
        print("[INFO] No RAG parquet found — generating demo RAG data")
        _, _, _, df_rag_long = make_demo_data(n_cases=df_v1_long["case_id"].nunique())

print("All data loaded")

In [ ]:
# ── Merge all sources into one master long DataFrame ─────────────────────────
df = df_v1_long.copy()
df = df.merge(df_v2_long, on=["case_id", "feature"], how="left")
df = df.merge(
    df_rag_long[["case_id", "feature", "verdict_rag"]].drop_duplicates(),
    on=["case_id", "feature"], how="left",
)
df = df.merge(df_doc_quality, on="case_id", how="left")

# Derived binary flags
df["fab_v1"]   = (df["status_v1"]  == 3).astype(int)
df["minor_v1"] = (df["status_v1"]  == 2).astype(int)
df["correct_v1"] = (df["status_v1"] == 1).astype(int)
df["absent_v1"]  = (df["status_v1"] == 0).astype(int)

df["fab_v2"]   = (df["status_v2"]  == 3).astype(float)
df["minor_v2"] = (df["status_v2"]  == 2).astype(float)
df["correct_v2"] = (df["status_v2"] == 1).astype(float)

df["fab_rag"]     = (df["verdict_rag"] == "FABRICATION").astype(float)
df["correct_rag"] = (df["verdict_rag"] == "CORRECT").astype(float)
df["uncertain_rag"] = (df["verdict_rag"] == "UNCERTAIN").astype(float)

df["domain"] = df["feature"].apply(
    lambda f: "Radiology" if f in IMAGING_FEATURES else "Pathology"
)
df["feature_label"] = df["feature"].map(FEAT_LABELS).fillna(df["feature"])

print(f"Master table: {df.shape}")
print(f"Cases: {df['case_id'].nunique()}   Features/case: {len(FEATURE_NAMES)}")
df.head(3)

---
## Section 2 · Source Document Quality Profile

In [ ]:
# ── Case-level quality summary table ─────────────────────────────────────────
# Join fabrication counts from v1
fab_per_case = df.groupby("case_id")["fab_v1"].sum().rename("n_fab_v1")
err_per_case = df.groupby("case_id")["minor_v1"].sum().rename("n_minor_v1")

dq_cols = [c for c in df_doc_quality.columns if c != "case_id"]
tbl = (
    df_doc_quality
    .set_index("case_id")
    .join(fab_per_case)
    .join(err_per_case)
    .reset_index()
)

QUAL_LABEL = {0: "Good", 1: "Possible concern", 2: "Confirmed issue"}
if "doc_quality_flag" in tbl.columns:
    tbl["quality_tier"] = tbl["doc_quality_flag"].map(QUAL_LABEL).fillna("Unknown")

display_cols = [c for c in [
    "case_id", "n_pdfs", "n_pages_total", "blur_mean", "contrast_mean",
    "doc_quality_flag", "quality_tier", "pdf_type",
    "word_count_src", "lexical_div_src", "n_fab_v1", "n_minor_v1",
] if c in tbl.columns]

print("=== Source Document Quality Profile ===")
print(tbl[display_cols].to_string(index=False))

# Aggregate by quality tier
if "quality_tier" in tbl.columns and "n_fab_v1" in tbl.columns:
    agg = tbl.groupby("quality_tier").agg(
        n_cases=("case_id", "count"),
        mean_blur=("blur_mean", "mean"),
        mean_contrast=("contrast_mean", "mean"),
        mean_fab=("n_fab_v1", "mean"),
    ).round(2).reset_index()
    print("\n=== Aggregated by Quality Tier ===")
    print(agg.to_string(index=False))

In [ ]:
# ── Doc quality distribution plots ───────────────────────────────────────────
has_blur     = "blur_mean"      in df_doc_quality.columns
has_contrast = "contrast_mean"  in df_doc_quality.columns
has_pages    = "n_pages_total"  in df_doc_quality.columns or "n_pages" in df_doc_quality.columns

blur_col  = "blur_mean"     if has_blur     else ("blur_laplacian" if "blur_laplacian" in df_doc_quality.columns else None)
cont_col  = "contrast_mean" if has_contrast else ("contrast_rms"   if "contrast_rms"   in df_doc_quality.columns else None)
page_col  = "n_pages_total" if "n_pages_total" in df_doc_quality.columns else ("n_pages" if "n_pages" in df_doc_quality.columns else None)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

if blur_col:
    axes[0].hist(df_doc_quality[blur_col].dropna(), bins=15, color="#3498db", edgecolor="white")
    axes[0].axvline(df_doc_quality[blur_col].median(), color="red", ls="--",
                    label=f"Median={df_doc_quality[blur_col].median():.0f}")
    axes[0].set_xlabel("Blur (Laplacian variance)")
    axes[0].set_ylabel("Cases")
    axes[0].set_title("Blur Score Distribution", fontweight="bold")
    axes[0].legend(fontsize=9)
    axes[0].axvspan(0, 50,  alpha=0.08, color="red",   label="Poor")
    axes[0].axvspan(50, 100, alpha=0.06, color="orange")

if cont_col:
    axes[1].hist(df_doc_quality[cont_col].dropna(), bins=15, color="#9b59b6", edgecolor="white")
    axes[1].axvline(df_doc_quality[cont_col].median(), color="red", ls="--",
                    label=f"Median={df_doc_quality[cont_col].median():.0f}")
    axes[1].set_xlabel("Contrast (RMS)")
    axes[1].set_title("Contrast Distribution", fontweight="bold")
    axes[1].legend(fontsize=9)

if page_col:
    axes[2].hist(df_doc_quality[page_col].dropna(), bins=15, color="#27ae60", edgecolor="white")
    axes[2].set_xlabel("Total pages per case")
    axes[2].set_title("Document Length Distribution", fontweight="bold")

plt.suptitle(
    f"Source Document Quality — Fabrication Cases  (n={df_doc_quality['case_id'].nunique()})",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "07_doc_quality_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Section 3 · Document Quality vs Fabrication Association

In [ ]:
# ── Scatter: blur / contrast vs fabrication count ────────────────────────────
if "n_fab_v1" not in tbl.columns:
    tbl = tbl.merge(fab_per_case.reset_index(), on="case_id", how="left")

pal = {"Good": "#2ecc71", "Possible concern": "#f39c12", "Confirmed issue": "#e74c3c",
       "Unknown": "#95a5a6"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, xcol, xlabel in [
    (axes[0], blur_col,  "Blur score (Laplacian variance)"),
    (axes[1], cont_col,  "Contrast score (RMS)"),
]:
    if xcol and xcol in tbl.columns:
        tier_col = "quality_tier" if "quality_tier" in tbl.columns else None
        colors   = tbl[tier_col].map(pal).fillna("#95a5a6") if tier_col else "steelblue"
        ax.scatter(tbl[xcol], tbl["n_fab_v1"], c=colors, s=80, edgecolors="white",
                   linewidths=0.5, alpha=0.85)

        # Regression line
        valid = tbl[[xcol, "n_fab_v1"]].dropna()
        if len(valid) > 2:
            slope, intercept, r, p, _ = stats.linregress(valid[xcol], valid["n_fab_v1"])
            xs = np.linspace(valid[xcol].min(), valid[xcol].max(), 100)
            ax.plot(xs, slope * xs + intercept, "--", color="#2c3e50", lw=1.5,
                    label=f"r={r:.2f}  p={p:.3f}")
            ax.legend(fontsize=9)

        ax.set_xlabel(xlabel)
        ax.set_ylabel("# Fabrications per case (v1)")
        ax.set_title(f"Doc quality vs fabrications — {xlabel.split('(')[0].strip()}",
                     fontweight="bold")

        # Legend for quality tiers
        if tier_col:
            handles = [mpatches.Patch(color=c, label=l) for l, c in pal.items()
                       if l in tbl[tier_col].values]
            ax.legend(handles=handles, fontsize=8, loc="upper right")

plt.suptitle("Source Document Quality vs Fabrication Count (v1 prompt)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "07_quality_vs_fabrication_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Box: fabrication count by quality tier ───────────────────────────────────
if "quality_tier" in tbl.columns and "n_fab_v1" in tbl.columns:
    order = ["Good", "Possible concern", "Confirmed issue"]
    order = [o for o in order if o in tbl["quality_tier"].values]

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(
        data=tbl, x="quality_tier", y="n_fab_v1", order=order,
        palette=[pal.get(o, "#95a5a6") for o in order],
        ax=ax, width=0.45, linewidth=1.5,
    )
    sns.stripplot(
        data=tbl, x="quality_tier", y="n_fab_v1", order=order,
        color="#2c3e50", alpha=0.6, size=5, jitter=True, ax=ax,
    )
    ax.set_xlabel("Document quality tier")
    ax.set_ylabel("# Fabrications per case (v1)")
    ax.set_title("Fabrication Count by Document Quality Tier  (v1 prompt)",
                 fontweight="bold")

    # Kruskal-Wallis test
    groups = [tbl.loc[tbl["quality_tier"] == o, "n_fab_v1"].dropna() for o in order]
    groups = [g for g in groups if len(g) > 1]
    if len(groups) >= 2:
        stat, pval = stats.kruskal(*groups)
        ax.text(0.98, 0.97, f"Kruskal-Wallis  H={stat:.2f}  p={pval:.3f}",
                transform=ax.transAxes, ha="right", va="top", fontsize=9,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.7))

    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "07_quality_tier_boxplot.png", dpi=150)
    plt.show()

---
## Section 4 · Text Metrics: v1 vs v2 Summaries

In [ ]:
# ── Load v1/v2 Phase-1 text stats (if available) ──────────────────────────────
phase1_csv = RUNS_DIR / "v1_v2_comparison" / "phase1_text_features.csv"

if phase1_csv.exists():
    df_text = pd.read_csv(phase1_csv)
    print(f"Loaded Phase-1 text features: {df_text.shape}")
else:
    # Synthesise from doc_quality metadata
    rng = np.random.default_rng(0)
    cases = df_doc_quality["case_id"].tolist()
    text_rows = []
    for cid in cases:
        base = rng.integers(400, 1800)
        text_rows.append({
            "case_id":          cid,
            "v1_word_count":    int(base),
            "v2_word_count":    int(base * rng.uniform(0.35, 0.65)),  # v2 JSON more compact
            "v1_lexical_div":   round(rng.uniform(0.38, 0.60), 3),
            "v2_lexical_div":   round(rng.uniform(0.45, 0.70), 3),
            "v1_char_count":    int(base * 5.4),
            "v2_char_count":    int(base * 5.4 * rng.uniform(0.35, 0.65)),
            "v1_sentence_count":int(base / 18),
            "v2_sentence_count":int(base * rng.uniform(0.35, 0.65) / 12),
        })
    df_text = pd.DataFrame(text_rows)
    print(f"[INFO] Synthesised text metrics for {len(df_text)} cases")

df_text.head(3)

In [ ]:
# ── Figure: v1 vs v2 text stats per case (Figure 5 analog — Phase 1) ─────────
n = len(df_text)
x = np.arange(n)
w = 0.35

case_labels = (
    df_text["case_id"].str[-6:].tolist()
    if "case_id" in df_text.columns
    else [f"{i}" for i in range(n)]
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, v1_col, v2_col, ylabel, title in [
    (axes[0], "v1_word_count",  "v2_word_count",  "Words",    "Word Count"),
    (axes[1], "v1_lexical_div", "v2_lexical_div",  "Unique/Total", "Lexical Diversity"),
    (axes[2], "v1_char_count",  "v2_char_count",  "Characters", "Character Count"),
]:
    v1_c = v1_col if v1_col in df_text.columns else v1_col.replace("_div", "_diversity")
    v2_c = v2_col if v2_col in df_text.columns else v2_col.replace("_div", "_diversity")

    if v1_c in df_text.columns and v2_c in df_text.columns:
        ax.bar(x - w/2, df_text[v1_c], w, label="v1 (narrative)",
               color="#3498db", edgecolor="white", alpha=0.9)
        ax.bar(x + w/2, df_text[v2_c], w, label="v2 (JSON)",
               color="#e67e22", edgecolor="white", alpha=0.9)
        ax.set_xticks(x)
        ax.set_xticklabels(case_labels, rotation=55, ha="right", fontsize=7)
        ax.set_title(title, fontweight="bold")
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)

plt.suptitle(
    "Text Metrics by Prompt Version — Fabrication Cases  (v1: narrative vs v2: structured JSON)",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "07_text_metrics_v1_v2.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Section 5 · Prompt Version Performance Table

Inspired by **Table 2** of arXiv:2510.16091 — macro-averaged metrics across prompting strategies.  
Adapted for extraction fidelity: `Correct Rate`, `Minor Error Rate`, `Fabrication Rate`.  
Precision analog = correct / (correct + fabrication); Recall analog = correct / all present features.

In [ ]:
# ── Macro-averaged performance per prompt version ─────────────────────────────
# Only count rows where feature is present in source (status != 0)
df_present = df[df["status_v1"] != 0].copy()

rows_perf = []

# v1
n = len(df_present)
rows_perf.append({
    "Prompt Version":     "v1 — Narrative (no constraints)",
    "Correct Rate (%)":   round(df_present["correct_v1"].mean() * 100, 1),
    "Minor Error Rate (%)":round(df_present["minor_v1"].mean() * 100, 1),
    "Fabrication Rate (%)":round(df_present["fab_v1"].mean() * 100, 1),
    "Precision":          round(df_present["correct_v1"].sum() /
                                max(df_present["correct_v1"].sum() +
                                    df_present["fab_v1"].sum(), 1), 3),
    "Recall":             round(df_present["correct_v1"].mean(), 3),
})

# v2
v2_pres = df_present[df_present["status_v2"].notna()]
if len(v2_pres) > 0:
    rows_perf.append({
        "Prompt Version":      "v2 — Structured JSON (explicit constraints)",
        "Correct Rate (%)":    round(v2_pres["correct_v2"].mean() * 100, 1),
        "Minor Error Rate (%)":round(v2_pres["minor_v2"].mean() * 100, 1),
        "Fabrication Rate (%)":round(v2_pres["fab_v2"].mean() * 100, 1),
        "Precision":           round(v2_pres["correct_v2"].sum() /
                                     max(v2_pres["correct_v2"].sum() +
                                         v2_pres["fab_v2"].sum(), 1), 3),
        "Recall":              round(v2_pres["correct_v2"].mean(), 3),
    })

# v3 RAG
rag_pres = df_present[df_present["verdict_rag"].notna()]
if len(rag_pres) > 0:
    rows_perf.append({
        "Prompt Version":      "v3 — RAG + Verification + Self-Consistency",
        "Correct Rate (%)":    round(rag_pres["correct_rag"].mean() * 100, 1),
        "Minor Error Rate (%)":round(rag_pres["uncertain_rag"].mean() * 100, 1),
        "Fabrication Rate (%)":round(rag_pres["fab_rag"].mean() * 100, 1),
        "Precision":           round(rag_pres["correct_rag"].sum() /
                                     max(rag_pres["correct_rag"].sum() +
                                         rag_pres["fab_rag"].sum(), 1), 3),
        "Recall":              round(rag_pres["correct_rag"].mean(), 3),
    })

df_perf = pd.DataFrame(rows_perf)
df_perf["F1"] = (
    2 * df_perf["Precision"] * df_perf["Recall"]
    / (df_perf["Precision"] + df_perf["Recall"]).clip(lower=1e-9)
).round(3)

print("\n=== Table 2 (adapted): Macro-Averaged Performance by Prompt Version ===")
print(df_perf.to_string(index=False))
df_perf.to_csv(REPORTS_DIR / "07_prompt_version_perf_table.csv", index=False)

In [ ]:
# ── Precision–Recall tradeoff plot (Figure 5 analog) ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: grouped bar — correct/minor/fab rate per version
versions = df_perf["Prompt Version"].apply(lambda v: v.split("—")[0].strip()).tolist()
x  = np.arange(len(versions))
w  = 0.22

ax = axes[0]
ax.bar(x - w,    df_perf["Correct Rate (%)"],     w, label="Correct",     color="#2ecc71", edgecolor="white")
ax.bar(x,        df_perf["Minor Error Rate (%)"],  w, label="Minor error", color="#f39c12", edgecolor="white")
ax.bar(x + w,    df_perf["Fabrication Rate (%)"],  w, label="Fabrication", color="#e74c3c", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(versions, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("%")
ax.set_title("Outcome rates by prompt version", fontweight="bold")
ax.legend(fontsize=9)
ax.set_ylim(0, 105)
for rect in ax.patches:
    h = rect.get_height()
    if h > 1:
        ax.text(rect.get_x() + rect.get_width()/2, h + 0.5,
                f"{h:.0f}", ha="center", va="bottom", fontsize=7)

# Right: precision–recall scatter
ax2 = axes[1]
colors_pr = ["#3498db", "#e67e22", "#9b59b6"]
for i, (_, row) in enumerate(df_perf.iterrows()):
    ax2.scatter(row["Recall"], row["Precision"], s=180,
                color=colors_pr[i % len(colors_pr)], zorder=5,
                label=versions[i], edgecolors="white", linewidths=1)
    ax2.annotate(
        f"F1={row['F1']:.3f}",
        (row["Recall"], row["Precision"]),
        textcoords="offset points", xytext=(6, 4), fontsize=8,
    )

# Iso-F1 curves
f1_levels = [0.7, 0.8, 0.9]
for f1 in f1_levels:
    p_range = np.linspace(0.01, 1.0, 200)
    r_range = f1 * p_range / np.clip(2 * p_range - f1, 1e-9, None)
    mask = (r_range >= 0) & (r_range <= 1.0)
    ax2.plot(r_range[mask], p_range[mask], "--", color="#bdc3c7", lw=1, alpha=0.7)
    ax2.text(r_range[mask][-1] + 0.01, p_range[mask][-1],
             f"F1={f1}", fontsize=7, color="#7f8c8d")

ax2.set_xlabel("Recall (correct / all present features)")
ax2.set_ylabel("Precision (correct / correct+fabricated)")
ax2.set_title("Precision–Recall tradeoff by prompt version", fontweight="bold")
ax2.set_xlim(0.3, 1.05)
ax2.set_ylim(0.3, 1.05)
ax2.legend(fontsize=8)

plt.suptitle("Prompt Version Performance — Extraction Fidelity", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "07_prompt_perf_precision_recall.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Section 6 · Feature × Version Heatmap

Adapted from **Table A.1** (arXiv:2510.16091 Appendix): evaluation metrics for each prompt–model combination.  
Here: fabrication rate (%) per clinical feature × prompt version.

In [ ]:
# ── Build pivot: feature × version → fabrication rate ────────────────────────
# v1
fab_v1_feat = (
    df_present.groupby("feature_label")["fab_v1"].mean().mul(100).rename("v1")
)

# v2
fab_v2_feat = (
    df_present[df_present["status_v2"].notna()]
    .groupby("feature_label")["fab_v2"].mean().mul(100).rename("v2")
    if "fab_v2" in df_present.columns else pd.Series(dtype=float, name="v2")
)

# v3 RAG
fab_rag_feat = (
    df_present[df_present["verdict_rag"].notna()]
    .groupby("feature_label")["fab_rag"].mean().mul(100).rename("v3 (RAG)")
    if "fab_rag" in df_present.columns else pd.Series(dtype=float, name="v3 (RAG)")
)

pivot = pd.concat([fab_v1_feat, fab_v2_feat, fab_rag_feat], axis=1).fillna(0)
pivot = pivot.sort_values("v1", ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    pivot, ax=ax,
    cmap="RdYlGn_r", vmin=0, vmax=pivot.values.max() * 1.1 or 20,
    annot=True, fmt=".1f", linewidths=0.4,
    cbar_kws={"label": "Fabrication rate (%)"},
)
ax.set_title(
    "Fabrication Rate (%) per Clinical Feature × Prompt Version\n"
    "(Table A.1 adapted — arXiv:2510.16091)",
    fontsize=11, fontweight="bold"
)
ax.set_xlabel("Prompt Version")
ax.set_ylabel("Clinical Feature")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "07_feature_version_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

pivot.round(1).to_csv(REPORTS_DIR / "07_feature_version_heatmap.csv")

---
## Section 7 · Grouped Bar — Fabrication Rate by Feature × Version

Adapted from **Figure 5** (arXiv:2510.16091): grouped bar chart of performance metrics per prompt type across LLMs.

In [ ]:
versions_present = [c for c in ["v1", "v2", "v3 (RAG)"] if c in pivot.columns]
n_feat = len(pivot)
x  = np.arange(n_feat)
w  = 0.25
vc = {"v1": "#e74c3c", "v2": "#f39c12", "v3 (RAG)": "#2ecc71"}

fig, ax = plt.subplots(figsize=(16, 5))
for i, ver in enumerate(versions_present):
    offset = (i - (len(versions_present) - 1) / 2) * w
    bars = ax.bar(x + offset, pivot[ver], w, label=ver,
                  color=vc.get(ver, "steelblue"), edgecolor="white", alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(pivot.index.tolist(), rotation=40, ha="right", fontsize=8)
ax.set_ylabel("Fabrication rate (%)")
ax.set_title(
    "Fabrication Rate by Clinical Feature × Prompt Version  (Figure 5 adapted)",
    fontsize=12, fontweight="bold"
)
ax.legend(title="Prompt version", fontsize=9)
ax.set_ylim(0, pivot.values.max() * 1.25 + 2)

# Colour-code x-axis labels by domain
for tick, feat_lbl in zip(ax.get_xticklabels(), pivot.index.tolist()):
    feat_key = {v: k for k, v in FEAT_LABELS.items()}.get(feat_lbl, "")
    tick.set_color("#2980b9" if feat_key in IMAGING_FEATURES else "#8e44ad")

# Domain legend patches
ax.legend(
    handles=[
        *[mpatches.Patch(color=vc.get(v, "gray"), label=v) for v in versions_present],
        mpatches.Patch(color="white", label=""),
        mpatches.Patch(color="#2980b9", label="Radiology feature"),
        mpatches.Patch(color="#8e44ad", label="Pathology feature"),
    ],
    fontsize=8, ncol=2,
)

plt.tight_layout()
plt.savefig(REPORTS_DIR / "07_grouped_bar_fab_by_feature_version.png",
            dpi=150, bbox_inches="tight")
plt.show()

---
## Section 8 · Correction Analysis — v1 → v2 → v3

In [ ]:
# ── Delta fabrication rate per feature ───────────────────────────────────────
corr = pivot.copy()
if "v2" in corr.columns:
    corr["Δ v1→v2"] = corr["v2"] - corr["v1"]
if "v3 (RAG)" in corr.columns and "v2" in corr.columns:
    corr["Δ v2→v3"] = corr["v3 (RAG)"] - corr["v2"]
if "v3 (RAG)" in corr.columns:
    corr["Δ v1→v3"] = corr["v3 (RAG)"] - corr["v1"]

corr = corr.sort_values("Δ v1→v3" if "Δ v1→v3" in corr.columns else "v1")
print("=== Correction Table — Δ Fabrication Rate per Feature ===")
print(corr.round(1).to_string())
corr.round(1).to_csv(REPORTS_DIR / "07_correction_delta_table.csv")

In [ ]:
# ── Waterfall / lollipop: improvement v1 → v3 ────────────────────────────────
delta_col = "Δ v1→v3" if "Δ v1→v3" in corr.columns else ("Δ v1→v2" if "Δ v1→v2" in corr.columns else None)

if delta_col:
    srt = corr[delta_col].sort_values()
    colors = ["#2ecc71" if v <= 0 else "#e74c3c" for v in srt]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.barh(srt.index, srt.values, color=colors, edgecolor="white", height=0.6)
    ax.axvline(0, color="#2c3e50", lw=1.2)
    ax.set_xlabel("Δ Fabrication rate (pp)  [negative = improvement]")
    ax.set_title(
        f"Fabrication Rate Change per Feature: {delta_col}\n"
        "(green = corrected, red = worsened)",
        fontweight="bold"
    )
    for bar, val in zip(bars, srt.values):
        ax.text(val + (0.2 if val >= 0 else -0.2), bar.get_y() + bar.get_height()/2,
                f"{val:+.1f}pp", va="center", ha="left" if val >= 0 else "right",
                fontsize=8)
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "07_correction_waterfall.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Wilcoxon signed-rank test: v1 vs v2 fabrication rates ────────────────────
# Tests whether per-case fabrication counts differ significantly across versions
print("=== Statistical Tests ===")

fab_v1_per_case = df_present.groupby("case_id")["fab_v1"].sum()

if "fab_v2" in df_present.columns:
    fab_v2_per_case = (
        df_present[df_present["status_v2"].notna()]
        .groupby("case_id")["fab_v2"].sum()
    )
    common = fab_v1_per_case.index.intersection(fab_v2_per_case.index)
    if len(common) >= 5:
        stat, p = stats.wilcoxon(fab_v1_per_case[common], fab_v2_per_case[common],
                                  alternative="greater", zero_method="wilcox")
        print(f"v1 vs v2 — Wilcoxon signed-rank (v1 > v2):  W={stat:.1f}  p={p:.4f}")
        print(f"  Median v1 fabs/case: {fab_v1_per_case[common].median():.1f}")
        print(f"  Median v2 fabs/case: {fab_v2_per_case[common].median():.1f}")

if "fab_rag" in df_present.columns:
    fab_rag_per_case = (
        df_present[df_present["verdict_rag"].notna()]
        .groupby("case_id")["fab_rag"].sum()
    )
    common_rag = fab_v1_per_case.index.intersection(fab_rag_per_case.index)
    if len(common_rag) >= 5:
        stat, p = stats.wilcoxon(fab_v1_per_case[common_rag], fab_rag_per_case[common_rag],
                                  alternative="greater", zero_method="wilcox")
        print(f"v1 vs v3(RAG) — Wilcoxon signed-rank (v1 > v3):  W={stat:.1f}  p={p:.4f}")
        print(f"  Median v1 fabs/case: {fab_v1_per_case[common_rag].median():.1f}")
        print(f"  Median v3 fabs/case: {fab_rag_per_case[common_rag].median():.1f}")

# Per-feature McNemar-style change table
if "fab_v2" in df_present.columns:
    both = df_present[df_present["status_v2"].notna()]
    corrected = ((both["fab_v1"] == 1) & (both["fab_v2"] == 0)).sum()
    regressed = ((both["fab_v1"] == 0) & (both["fab_v2"] == 1)).sum()
    both_fab  = ((both["fab_v1"] == 1) & (both["fab_v2"] == 1)).sum()
    both_ok   = ((both["fab_v1"] == 0) & (both["fab_v2"] == 0)).sum()

    print(f"\n=== McNemar Contingency Table (v1 vs v2 fabrication) ===")
    print(f"  Corrected  (v1=fab, v2=ok) : {corrected}")
    print(f"  Regressed  (v1=ok,  v2=fab): {regressed}")
    print(f"  Persistent (both fab)      : {both_fab}")
    print(f"  Both OK                    : {both_ok}")
    if corrected + regressed >= 5:
        _, p_mc = stats.binom_test(corrected, corrected + regressed, p=0.5,
                                    alternative="greater") if hasattr(stats, 'binom_test') else (0, None)
        if p_mc is not None:
            print(f"  McNemar exact p (corrected > regressed): {p_mc:.4f}")

---
## Section 9 · Summary Dashboard

In [ ]:
# ── 2×2 summary dashboard ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

# (0,0) — Performance table as text plot
ax00 = fig.add_subplot(gs[0, 0])
ax00.axis("off")
tbl_data = df_perf[["Prompt Version", "Correct Rate (%)",
                     "Fabrication Rate (%)", "Precision", "Recall", "F1"]].copy()
tbl_data["Prompt Version"] = tbl_data["Prompt Version"].apply(
    lambda s: s.split("—")[0].strip() + "\n" + s.split("—")[-1].strip()
)
table = ax00.table(
    cellText=tbl_data.values, colLabels=tbl_data.columns,
    cellLoc="center", loc="center", bbox=[0, 0, 1, 1],
)
table.auto_set_font_size(False)
table.set_fontsize(7.5)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor("#2c3e50")
        cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#f8f9fa")
ax00.set_title("Table 2 — Prompt Version Performance", fontweight="bold", pad=8)

# (0,1) — Fabrication rate per version (bar)
ax01 = fig.add_subplot(gs[0, 1])
vers_short = [v.split("—")[0].strip() for v in df_perf["Prompt Version"]]
bars = ax01.bar(vers_short, df_perf["Fabrication Rate (%)"],
                color=["#e74c3c", "#f39c12", "#2ecc71"][:len(df_perf)],
                edgecolor="white", width=0.5)
ax01.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
ax01.set_ylabel("Fabrication rate (%)")
ax01.set_title("Fabrication Rate by Prompt Version", fontweight="bold")
ax01.set_ylim(0, df_perf["Fabrication Rate (%)"].max() * 1.4 + 2)
ax01.tick_params(axis="x", rotation=15)

# (1,0) — Feature heatmap (v1 only if others absent)
ax10 = fig.add_subplot(gs[1, 0])
heat_data = pivot[[c for c in ["v1", "v2", "v3 (RAG)"] if c in pivot.columns]].head(14)
sns.heatmap(
    heat_data, ax=ax10, cmap="RdYlGn_r", vmin=0,
    vmax=max(heat_data.values.max() * 1.05, 5),
    annot=True, fmt=".0f", linewidths=0.3, cbar_kws={"shrink": 0.8},
    annot_kws={"size": 7},
)
ax10.set_title("Fabrication Rate (%) by Feature × Version", fontweight="bold")
ax10.set_xlabel("")
ax10.tick_params(axis="both", labelsize=7)

# (1,1) — Correction waterfall if delta available
ax11 = fig.add_subplot(gs[1, 1])
delta_col = "Δ v1→v3" if "Δ v1→v3" in corr.columns else ("Δ v1→v2" if "Δ v1→v2" in corr.columns else None)
if delta_col:
    srt2 = corr[delta_col].sort_values()
    colors2 = ["#2ecc71" if v <= 0 else "#e74c3c" for v in srt2]
    ax11.barh(range(len(srt2)), srt2.values, color=colors2, edgecolor="white", height=0.65)
    ax11.set_yticks(range(len(srt2)))
    ax11.set_yticklabels(srt2.index.tolist(), fontsize=7)
    ax11.axvline(0, color="#2c3e50", lw=1.2)
    ax11.set_xlabel("Δ pp")
    ax11.set_title(f"Fabrication Correction ({delta_col})", fontweight="bold")
else:
    ax11.text(0.5, 0.5, "Δ data unavailable", ha="center", va="center",
              transform=ax11.transAxes, fontsize=12, color="gray")
    ax11.axis("off")

fig.suptitle(
    "Fabrication Analysis — Text & Doc Metrics × Prompt Version  (MSK Breast Oncology)",
    fontsize=13, fontweight="bold", y=0.98,
)
plt.savefig(REPORTS_DIR / "07_summary_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Dashboard saved → {REPORTS_DIR / '07_summary_dashboard.png'}")

---
## Section 10 · Export All Outputs

In [ ]:
outputs = {
    "07_master_long.csv":              df,
    "07_doc_quality_case.csv":         tbl[display_cols] if display_cols else df_doc_quality,
    "07_text_metrics_v1_v2.csv":       df_text,
    "07_prompt_version_perf_table.csv":df_perf,
    "07_feature_version_heatmap.csv":  pivot.round(1),
    "07_correction_delta_table.csv":   corr.round(1),
}

for fname, data in outputs.items():
    path = REPORTS_DIR / fname
    try:
        data.to_csv(path, index=fname in ("07_feature_version_heatmap.csv",
                                           "07_correction_delta_table.csv"))
        print(f"Saved: {path.name}  ({len(data)} rows)")
    except Exception as e:
        print(f"[WARN] Could not save {fname}: {e}")

print("\n=== Figures generated ===")
for f in sorted(REPORTS_DIR.glob("07_*.png")):
    print(f"  {f.name}")